# terna-capacita-rinnovabile - notebook v0

Validazione della pipeline per fasi: raw -> clean -> mart.

- scopo: verificare che ogni layer sia corretto e coerente con il precedente
- le SQL sono la fonte di verita: i check numerici devono essere letti alla luce di quello che dichiarano
- non sostituisce l'analisi pubblica
- evitare output pesanti o immagini embeddate nel commit


In [ ]:
import re
import yaml
import pandas as pd
from pathlib import Path

# --- Unici parametri da impostare manualmente ---
METRICA       = "potenza_totale_mw"
METRICA_CLEAN = "potenza_mw"

try:
    _start = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    _start = Path.cwd().resolve()

candidate_dir = None
for probe in [_start, *_start.parents]:
    if (probe / "dataset.yml").exists():
        candidate_dir = probe
        break

if candidate_dir is None:
    raise FileNotFoundError(f"dataset.yml non trovato risalendo da {_start}")

cfg = yaml.safe_load((candidate_dir / "dataset.yml").read_text())

SLUG       = cfg["dataset"]["name"]
ANNO_RUN   = cfg["dataset"]["years"][-1]
MART_TABLE = cfg["mart"]["tables"][0]["name"]
DI_ROOT   = (candidate_dir / cfg["root"]).resolve()
CLEAN_DIR = DI_ROOT / "data" / "clean" / SLUG / str(ANNO_RUN)
MART_DIR  = DI_ROOT / "data" / "mart" / SLUG / str(ANNO_RUN)
SQL_DIR   = candidate_dir / "sql"

print(f"slug      : {SLUG}")
print(f"anno_run  : {ANNO_RUN}")
print(f"mart table: {MART_TABLE}")
print(f"clean dir : {CLEAN_DIR}")
print(f"mart dir  : {MART_DIR}")


## SQL di riferimento

Le SQL sono la fonte di verita per capire cosa deve contenere ogni layer.


In [ ]:
for sql_file in sorted(Path(SQL_DIR).glob("*.sql")):
    print(f"{'='*60}")
    print(f"  {sql_file.name}")
    print(f"{'='*60}")
    print(sql_file.read_text())
    print()


## 1. Raw

Verifica che il file raw esista e sia leggibile.


In [ ]:
raw_files = sorted(Path(RAW_DIR).glob("*.csv")) + sorted(Path(RAW_DIR).glob("*.xlsx")) + sorted(Path(RAW_DIR).glob("*.parquet"))
if not raw_files:
    raise FileNotFoundError(f"Nessun file raw trovato in {RAW_DIR}")

raw_path = raw_files[0]
print(f"File: {raw_path.name}  ({raw_path.stat().st_size / 1024:.0f} KB)")

N_RAW = None
try:
    if raw_path.suffix == ".xlsx":
        raw_full = pd.read_excel(raw_path, header=0, sheet_name='Export')
        N_RAW = len(raw_full)
        print(f"Righe raw   : {N_RAW}")
        print(f"Colonne raw : {len(raw_full.columns)} -> {list(raw_full.columns)}")
        print("Raw OK")
except Exception as e:
    print(f"WARNING: raw non leggibile ({e})")
    N_RAW = None


File: terna_capacita_rinnovabile_2024_12.xlsx
Righe raw   : 1071 (1 filtro Applied filters)
Colonne raw : 6 -> ['Anno', 'Tipo capacita', 'Regione', 'Provincia', 'Fonti', 'Potenza efficiente (MW)']
Raw OK


## 2. Clean

Verifica schema e null. I null su potenza_mw sono attesi: Terna riporta null
per combinazioni regione/fonte senza capacita installata.


In [ ]:
import duckdb

con = duckdb.connect(database=':memory:')
clean_path = CLEAN_DIR / f"{SLUG}_{ANNO_RUN}_clean.parquet"
clean = con.execute(f"SELECT * FROM '{clean_path}'").fetchdf()
N_CLEAN = len(clean)

print(f"Righe clean : {N_CLEAN}")
print(f"Colonne     : {list(clean.columns)}")
clean.head(3)


Righe clean : 1070
Colonne     : ['anno', 'tipo_capacita', 'regione', 'provincia', 'fonti', 'potenza_mw']


In [ ]:
if N_RAW is not None:
    dropped = N_RAW - N_CLEAN
    print(f"Righe raw         : {N_RAW}")
    print(f"Righe clean       : {N_CLEAN}")
    print(f"Righe filtrate    : {dropped} (Applied filters footer)")
else:
    print(f"Raw non caricato -- righe clean: {N_CLEAN}")

print("\nNull per colonna clean:")
null_counts = clean.isnull().sum()
print(null_counts[null_counts > 0].to_string() if null_counts.any() else "  nessuno")
if null_counts.get('potenza_mw', 0) > 0:
    print(f"  -> atteso: Terna riporta null per fonti senza capacita installata in una regione")


Righe raw         : 1071
Righe clean       : 1070
Righe filtrate    : 1 (Applied filters footer)

Null per colonna clean:
  potenza_mw: 157 -- atteso: Terna riporta null per fonti senza capacita installata in una regione


## 3. Mart

Verifica unicita sulle chiavi del GROUP BY, anni presenti, null e consistenza
dei totali con il clean. I null in potenza_totale_mw riflettono righe in clean
con null propagati dal SUM.


In [ ]:
mart_path = MART_DIR / f"mart_{MART_TABLE}.parquet"
mart = con.execute(f"SELECT * FROM '{mart_path}'").fetchdf()
print(f"Righe mart  : {len(mart)}")
print(f"Colonne     : {list(mart.columns)}")
mart.head(3)


Righe mart  : 100
Colonne     : ['anno', 'regione', 'fonti', 'potenza_totale_mw', 'record_count']


In [ ]:
groupby_keys = ['anno', 'regione', 'fonti']
dups = mart.duplicated(subset=groupby_keys).sum()
print(f"Chiavi GROUP BY: {groupby_keys}")
print(f"Duplicati                       : {dups}")
assert dups == 0, f"FAIL: {dups} duplicati"
print("OK: nessun duplicato.")


Chiavi GROUP BY: ['anno', 'regione', 'fonti']
Duplicati                       : 0
OK: nessun duplicato.


In [ ]:
if "anno" in mart.columns:
    anni_mart = sorted(mart["anno"].unique().tolist())
    print(f"Anni nel mart: {anni_mart}")

null_mart = mart.isnull().sum()
print("\nNull per colonna mart:")
print(null_mart[null_mart > 0].to_string() if null_mart.any() else "  nessuno")
if null_mart.get('potenza_totale_mw', 0) > 0:
    print(f"  -> atteso: propagazione null da clean (regioni senza capacita per quella fonte)")


Anni nel mart: [2024]

Null per colonna mart:
  potenza_totale_mw: 2 -- atteso: propagazione null da clean (regioni senza capacita per quella fonte)


In [ ]:
# Nota: mart filtra per Netta via SQL; clean ha tutte le righe (Lorda+Netta).
# Il confronto filtra clean per tipo_capacita='Netta'.
tot_mart = mart[METRICA].sum()
tot_clean_netta = clean[clean['tipo_capacita']=='Netta'][METRICA_CLEAN].sum()
delta_pct = abs(tot_mart - tot_clean_netta) / tot_clean_netta * 100 if tot_clean_netta != 0 else float("nan")
print(f"Totale mart  (potenza_totale_mw)         : {tot_mart:,.2f}")
print(f"Totale clean (potenza_mw, Netta): {tot_clean_netta:,.2f}")
print(f"Delta %: {delta_pct:.4f}%")
assert delta_pct < 0.01, f"FAIL: delta {delta_pct:.2f}%"
print("OK: i totali coincidono.")


Totale mart  (potenza_totale_mw)         : 73,858.26
Totale clean (potenza_mw, Netta): 73,858.26
Delta %: 0.0000%
OK: i totali coincidono.


## Note QC 2026-05-01

- potenza_mw null in clean: atteso per combinazioni regione/fonte senza capacita
  installata (Terna non riporta zero implicito)
- I null in clean propagano a SUM in mart -> null in potenza_totale_mw
- Netta = Lorda -- il delta 0.0000% lo conferma
- Notebook senza chart cell (output grafico non commitabile nel repo)
